# Week 8: Clustering and Principal Component Analysis 

Lecture in week 6 and week 8 (see the corresponding lecture notes and text books)

The dataset and part of the exercises for this week were taken from MLPA.

<hr style="border:2px solid gray">

## Index: <a id='index'></a>
1. [Clustering](#clustering)
2. [PCA](#pca)


<hr style="border:2px solid gray">

## Clustering <a id='clustering'></a>


### K-means
K-means is a popular unsupervised machine learning algorithm for clustering.  K-means works by grouping similar data points together, aiming to minimise the distance within each cluster.

Here's a breakdown of how K-means clustering works:

* **Choosing the number of clusters (k):** This is a crucial step. You need to specify the desired number of clusters (k) beforehand. There are techniques like the elbow method to help you decide on an appropriate k value.
* **Initializing centroids:** K-means assigns data points to clusters based on their proximity to a representative point within the cluster, called the centroid. The algorithm starts by randomly placing k centroids within the data space.
* **Assigning data points:** Each data point is assigned to the cluster with the closest centroid based on a distance measure, typically Euclidean distance. 
* **Recomputing centroids:** Once all data points are assigned, the centroids are recomputed as the mean of the data points within each cluster.
* **Repeating the process:** This assignment and centroid update process iterates until a stopping criterion is met. The common criterion is that the centroids no longer move significantly between iterations, indicating convergence.

Here are some key things to remember about K-means:

* **Strengths:** K-means is a simple, efficient algorithm that works well for spherical-shaped clusters in the data. It's also easy to understand and implement.
* **Weaknesses:** K-means requires pre-specifying the number of clusters (k), which can be challenging. It also performs poorly with non-spherical clusters and can be sensitive to outliers in the data.
* **Applications:** K-means has various applications, including customer segmentation, image segmentation, and anomaly detection.

By understanding K-means, you can leverage its strengths for exploratory data analysis and identify underlying patterns within your unlabeled data. However, it's important to be aware of its limitations and choose the right approach based on your specific data and clustering goals.

In [2]:
import numpy as np
import pandas as pd
import os
import random
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
from sklearn import metrics

from sklearn import preprocessing, decomposition
import skimage
from skimage.transform import resize, rescale
from skimage import io


ModuleNotFoundError: No module named 'skimage'

In [ ]:
from mlxtend.plotting import plot_decision_regions

The function `make_blobs` from `sklearn` generates isotropic Gaussian distributions. Run the cell below to generate of sample of 300 instances around 4 centres, with a standard deviation of 0.6.

In [ ]:
# Generate synthetic data using make_blobs.
X, y_true = make_blobs(
    n_samples=300,  # Total number of points equally divided among clusters.
    centers=4,      # Number of centres to generate, or the number of clusters.
    cluster_std=0.6, # Standard deviation of the clusters.
    random_state=2   # Determines random number generation for dataset creation. Ensures reproducibility.
)

Plot a scatter plot of the generated blobs:

In [ ]:
# Create a scatter plot of the generated data (option s is for the size of the markers, and c for the colour)
plt.scatter( X[:, 0], X[:, 1], s=3, c='gray')

Before using the kmeans algorithm, we start with a rough position of the centers to start.

In [ ]:
start = np.array([[-1,1],[1,-1],[3,-3],[-5,-10]]) 

Now run the cell below to see how `kmeans` gets converges to the right clusters:

In [ ]:

fig = plt.figure(figsize=(12,6))

plt.subplot(2,2,1)
plt.scatter(X[:, 0], X[:, 1], s =3, c ='gray') # plot original points
plt.scatter(start[:,0],start[:,1], s = 20, c = 'k', label = 'Iteration 0') # plot the rough position of the centres
plt.xlim(-10,5);
plt.annotate('Iteration 0', xy=(77, 20), xycoords='axes points',
            size=14, ha='right', va='top')

for i in range(1,4):
    plt.subplot(2,2,i+1)

    # Initialise KMeans with 4 clusters, max_iter set to the current iteration, initial centroids 'start'
    # and n_init=1 (this is the number of times KMeans is run with different seeds, for reproducibility we use 1
    kmeans = KMeans(n_clusters=4, max_iter = i, init = start, n_init=1)
    
    kmeans.fit(X)
    y_kmeans = kmeans.predict(X)
    centroids = kmeans.cluster_centers_

    # Plot the data points coloured by their predicted cluster labels,
    plt.scatter(X[:, 0], X[:, 1], s = 3, c = y_kmeans, cmap = 'rainbow') 

    # Plot the centroids positions
    plt.scatter(centroids[:, 0], centroids[:, 1], s=20, \
                edgecolor = 'k', label = 'Iteration'+str(i), c = [0,1,2,3],cmap = 'rainbow');
    plt.xlim(-10,5);
    plt.annotate('Iteration '+str(i), xy=(77, 20), xycoords='axes points',
            size=14, ha='right', va='top')


<div style="background-color:#C2F5DD">

## Exercise 1
1. Change the start vector to a bad position (`start = np.array([[-6,2.5],[-5.5,2.5],[-0.5,1],[1,-1]])`
2. Verify with plots similar to the ones we made above if the algorithm converges after 4 iterations?
3. What about 10?

### Limitations of k-means
With a bad starting point, kMeans struggles. Let's take a look at another tricky example: overlapping blobs of different size/density.

In [ ]:
X1b, y1b = make_blobs(n_samples=200, centers=[(1.25,1)],
                       cluster_std=0.2, random_state=1)

X2b, y2b = make_blobs(n_samples=400, centers=[(0,1)],
                       cluster_std=0.5, random_state=2)

X3b, y3b = make_blobs(n_samples=200, centers=[(-1.25,1)],
                       cluster_std=0.2, random_state=3)

In [ ]:
fig = plt.figure(figsize=(12,6))

plt.scatter(X1b[:, 0], X1b[:, 1], s =10, c ='gray')

plt.scatter(X2b[:, 0], X2b[:, 1], s =10, c ='violet') 

plt.scatter(X3b[:, 0], X3b[:, 1], s =10, c ='teal')

Combine clusters in single data set:

In [ ]:
Xb = np.vstack([X1b,X2b,X3b])

Fixing the random seed generates consistent predictions (the predictions of the cluster assignments are usually consistent, but which cluster is predicted as 0, which as 1, which as 2 would otherwise vary.)

In [ ]:
kmeans = KMeans(n_clusters=3, n_init = 10, random_state=30) #predicts 0,1,2
kmeans.fit(Xb)
yb_kmeans = kmeans.predict(Xb)
centersb = kmeans.cluster_centers_

In [ ]:
yb_kmeans

Create vector of labels with the same convention for verification.

In [ ]:
yb = np.concatenate([np.zeros(len(y1b)),np.zeros(len(y2b))+1,np.zeros(len(y3b))+2])

Now we can compare the cluster assignments from the k means algorithm to the original ones.

In [ ]:
plt.figure(figsize=(8,6))
model = KMeans(n_clusters=3, n_init = 10, random_state=30) #predicts 0,1,2
model.fit(Xb)


# Plot the decision regions of the KMeans model.
# Xb: The input data.
# yb.astype(int): The true labels (converted to integers) for colouring the decision regions.
# clf=model: The trained KMeans model.
# legend=0: Do not show the legend.
# markers='...': Use dots as markers for the decision regions.
# colors='lightgray,violet,teal': Use specified colours for the decision regions.
plot_decision_regions(Xb, yb.astype(int), clf = model, legend=0, markers = '...', colors = 'lightgray,violet,teal')

plt.scatter(X1b[:,0],X1b[:,1], s = 30, c = 'lightgray',edgecolors='k')
plt.scatter(X2b[:,0],X2b[:,1], s = 30, c = 'violet', edgecolors='k')
plt.scatter(X3b[:,0],X3b[:,1],s = 30, c = 'teal', edgecolors='k')
plt.scatter(centersb[:, 0], centersb[:, 1], c='black', s=100, alpha=0.5);

plt.xlim(-2.5,2.5)
plt.ylim(-0.5,2.5);


### The elbow curve

The elbow curve is a visual tool used in clustering, specifically to determine the optimal number of clusters (k) for algorithms like K-means. We are looking for the largest decrease in the loss function.


In [ ]:
inertiasb = []
for k in range(2, 10):
    kmeans = KMeans(n_clusters=k, n_init = 10)
    kmeans.fit(Xb)
    inertiasb.append(kmeans.inertia_)

In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.plot(range(2, 10), inertiasb)
plt.grid(True)
plt.title('Elbow curve for blobs');
plt.xlabel('Number of clusters $k$', fontsize = 14);
plt.ylabel('$k$-means cost function', fontsize = 14);


### Silhouette score

The silhouette score is another metric used to evaluate the success of a clustering method and pick a number of clusters. Higher values correspond to better-defined clustering schemes.

In [ ]:
from sklearn.metrics import silhouette_samples, silhouette_score

In [ ]:
n_clusters = np.arange(2, 6)

for n in n_clusters:
    
    model = KMeans(n_clusters = n, n_init = 10, random_state = 10)

    model.fit(Xb)

    y_kmeans = model.predict(Xb)
    
    # Calculate the silhouette scores for each sample.
    silhouette_scores = silhouette_samples(Xb, y_kmeans)

    xlower = 10

    # Create a figure with two subplots: one for scatter plot of clustered data axs[0], and one for silhouette plot axs[1].
    fig, axs = plt.subplots(1, 2, figsize=(16, 8))
    ax = axs[1]

    # Assign colours to each data point based on its cluster assignment.
    colors = plt.cm.Accent(y_kmeans.astype(float)/n)
    ax.scatter(Xb[:, 0], Xb[:, 1], c=colors, s=40,edgecolor='k');

    # Adjust tick label size for better readability.
    ax.tick_params(axis='both', which='both', labelsize=20);
    ax = axs[0]

    # Iterate through each unique cluster label.
    for i in np.unique(y_kmeans):

        # Get the indices of data points belonging to the current cluster.
        ind = y_kmeans==i
        
        # Sort the silhouette scores for the current cluster.
        silh = np.sort(silhouette_scores[ind])
        
        # Get the size of the current cluster.
        size_cluster_i = silh.shape[0]
        
        # Calculate the upper x-axis limit for the current cluster's silhouette plot.
        xupper = xlower + size_cluster_i

        # Assign a color to the current cluster.
        color = plt.cm.Accent(float(i)/model.n_clusters)
   
        # Fill the area under the silhouette scores for the current cluster.
        ax.fill_between(np.arange(xlower, xupper), 0, silh, facecolor=color, edgecolor=color, alpha=0.7)
        
        # Draw a horizontal line at y=0.
        ax.axhline(y=0, c='k', lw=2)
        
        # Add text annotations
        ax.text(0.05, 0.95, '%0.0f clusters'%n, transform=ax.transAxes, fontsize=20)
        ax.text(0.45, 0.95, 'Mean S. score: %0.2f'%np.mean(silhouette_scores), transform=ax.transAxes, fontsize=20)

        # Update the lower x-axis limit for the next cluster.
        xlower = xupper + 10
        
        ax.set_ylabel('Silhouette score', fontsize=16)
        ax.set_ylim(-0.15,0.85)
        
    # Draw a horizontal line at the mean silhouette score for all clusters.
    ax.axhline(y=np.mean(silhouette_scores), color="red", linestyle="--")
    # Adjust tick label size.
    ax.tick_params(axis='both', which='both', labelsize=20);
    # Remove x-axis ticks.
    ax.set_xticks([]);

### Now we move on to a different distribution (smiley face).

Generate points for the smiley face distribution:

In [ ]:
from math import pi, cos, sin
from random import random

def point(h, k, r):
    theta = random() * 2 * pi
    return h + cos(theta) * r, k + sin(theta) * r + 0.2*random()

xy = [point(1,2,1) for _ in range(100)]

In [ ]:
X1, y1 = make_blobs(n_samples=10, centers=[(0.5,2.5)],
                       cluster_std=0.05, random_state=1)

X2, y2 = make_blobs(n_samples=10, centers=[(1.5,2.5)],
                       cluster_std=0.05, random_state=2)

X3, y3 = make_blobs(n_samples=10, centers=[(1,1.7)],
                       cluster_std=0.05, random_state=2)

In [ ]:
X3_stretch = np.array([X3[:,0]*3, X3[:,1]]) #make the mouth :) 

In [ ]:
plt.axes().set_aspect('equal', 'datalim')
plt.scatter(*zip(*xy))
plt.scatter(X1[:,0],X1[:,1])
plt.scatter(X2[:,0],X2[:,1])
plt.scatter(X3_stretch.T[:,0]-1.9,X3_stretch.T[:,1])

plt.show()

Finally, we can put together the four sets of points in the figure to create the data set as a single array:


In [ ]:
X = np.vstack([xy,X1,X2,np.array([X3_stretch.T[:,0]-1.9,X3_stretch.T[:,1]]).T])

<div style="background-color:#C2F5DD">

## Exercise 2
1. Run kMeans for 4 clusters and 10 iterations (random_state=32).
2. Plot the results in a scatter plot (predictions in the c colour) and plot the centres found in the step above.
3. Create a vector with the true labels (like we did above).
4. Make a second pot to compare the clustering assignments of k means (background colours) with the true labels.
5. Calculate and visualise the elbow curve.
6. Calculate and visualise the silhouette curve.


## Density-based clustering methods, DBSCAN and OPTICS.

### DBScan

DBScan, which stands for Density-Based Spatial Clustering of Applications with Noise, is another unsupervised learning algorithm used for clustering in machine learning. Unlike K-means, DBScan doesn't require you to pre-specify the number of clusters. Instead, it focuses on identifying high-density regions of data points, separated by areas of lower density.

Here's a deeper look at how DBScan works:

* **Core points, border points, and noise:** DBScan classifies data points into three categories:
    * Core points: These are densely surrounded by other points within a specific radius (epsilon) and have a minimum number of neighbours (MinPts). They form the core of the clusters.
    * Border points: They lie on the edges of clusters, having less than MinPts neighbours within epsilon but our neighbours to core points.
    * Noise: These are points that are isolated and don't meet the criteria for core or border points. They are considered outliers.
* **Density-based clustering:** DBScan iteratively examines each data point. If it's a core point, it expands the cluster by considering its neighbours within epsilon. If those neighbours are also core points, they are added to the cluster, and the process continues for their neighbours as well, forming a connected dense region. Border points are essentially attached to these core-point based clusters.
* **Handling noise:** Data points that are neither core points nor border points (have few neighbours) are classified as noise. DBScan effectively separates these outliers from the main clusters.

Here are some key advantages of DBScan:

* **No need for pre-defined clusters:** Unlike K-means, you don't have to specify the number of clusters beforehand. DBScan automatically discovers clusters based on density.
* **Robust to outliers:** DBScan can effectively identify and isolate outliers in the data as noise points.
* **Flexible cluster shapes:**  It can handle clusters of arbitrary shapes, not limited to spherical clusters like K-means.

However, DBScan also has some limitations:

* **Parameter tuning:** Although it doesn't require specifying cluster numbers, DBScan uses two parameters: epsilon (density threshold) and MinPts (minimum neighbours). Choosing appropriate values can be crucial for optimal clustering and can involve some trial and error.
* **Computational cost:** Compared to K-means, DBScan can be computationally more expensive, especially for very large datasets.

Overall, DBScan offers a powerful alternative to K-means for clustering tasks. Its ability to handle unknown cluster shapes and outliers makes it a valuable tool for various data analysis scenarios.


In [ ]:
from sklearn.cluster import DBSCAN

#Code adapted from: https://scikit-learn.org/stable/auto_examples/cluster/plot_dbscan.html

From the sklearn docs:

The DBSCAN algorithm views clusters as areas of high density separated by areas of low density. Due to this rather generic view, clusters found by DBSCAN can be any shape, as opposed to k-means which assumes that clusters are convex shaped. The central component to the DBSCAN is the concept of core samples, which are samples that are in areas of high density. A cluster is therefore a set of core samples, each close to each other (measured by some distance measure) and a set of non-core samples that are close to a core sample (but are not themselves core samples). 

Two Key Parameters:
```
eps (epsilon): defines the radius of the neighbourhood around a point. Points within this radius are considered neighbours.
min_samples: specifies the minimum number of points required within the eps neighbourhood for a point to be considered a core point.
```

min_samples and eps define what we mean when we say dense. Higher min_samples or lower eps indicate higher density necessary to form a cluster.

More formally, we define a core sample as being a sample in the dataset such that there exist min_samples other samples within a distance of eps, which are defined as neighbors of the core sample. This tells us that the core sample is in a dense area of the vector space. A cluster is a set of core samples that can be built by recursively taking a core sample, finding all of its neighbors that are core samples, finding all of their neighbors that are core samples, and so on. A cluster also has a set of non-core samples, which are samples that are neighbors of a core sample in the cluster but are not themselves core samples. Intuitively, these samples are on the fringes of a cluster.

Any core sample is part of a cluster, by definition. Any sample that is not a core sample, and is at least eps in distance from any core sample, is considered an outlier by the algorithm.

In [ ]:
# Initialize and fit the DBSCAN model with eps=0.25, min_samples=2
db = DBSCAN(eps=0.25, min_samples=2).fit(X)

# Create a boolean mask to identify core samples.
core_samples_mask = np.zeros_like(db.labels_, dtype=bool)
core_samples_mask[db.core_sample_indices_] = True

# Extract the cluster labels assigned by DBSCAN.
labels = db.labels_

# Calculate the number of unique clusters, excluding noise (-1 label).
n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)

# Calculate the number of noise points (-1 labels).
n_noise_ = list(labels).count(-1)

# Print the estimated number of clusters and noise points.
print('Estimated number of clusters: %d' % n_clusters_)
print('Estimated number of noise points: %d' % n_noise_)



In [ ]:
# #############################################################################
# Plot the results

# Get the set of unique labels (including noise).
unique_labels = set(labels)

# Generate a list of colors for each unique label.
colors = [plt.cm.Spectral(each)
          for each in np.linspace(0, 1, len(unique_labels))]

# Iterate through each unique label and its corresponding color.
for k, col in zip(unique_labels, colors):
    # If the label is -1, it's noise, so use black color.
    if k == -1:
        # Black used for noise.
        col = [0, 0, 0, 1]

    # Create a boolean mask to identify points belonging to the current cluster.
    class_member_mask = (labels == k)

    # Plot the core samples of the current cluster.
    xy = X[class_member_mask & core_samples_mask]
    plt.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col),
             markeredgecolor='k', markersize=14)

    # Plot the border/non-core samples of the current cluster.
    xy = X[class_member_mask & ~core_samples_mask]
    plt.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col),
             markeredgecolor='k', markersize=6)

# Set the title of the plot.
plt.title('Estimated number of clusters: %d' % n_clusters_)

# Display the plot.
plt.show()

We can see how the clustering changes as we vary the distance parameter, eps:

### OPTICS

The OPTICS (Ordering Points To Identify the Clustering Structure) algorithm is a density-based clustering algorithm. It's closely related to DBSCAN, but it addresses one of DBSCAN's limitations: handling clusters of varying densities.   

Unlike DBSCAN, which requires a single eps (epsilon) parameter for the neighbourhood radius, OPTICS creates an ordering of the data points representing the density-based clustering structure.
This ordering allows you to extract clusters of different densities from the same dataset.   

OPTICS produces a "reachability plot" that visualises the density structure of the data.   
This plot shows how close each point is to its neighbours, providing insights into the varying densities of the clusters.   

Key parameters in the implementation are:

```
min_samples: The minimum number of samples required to form a core point.
max_eps: The maximum distance between two samples for one to be considered as in the neighborhood of the other.
xi: Determines the minimum steepness on the reachability plot that constitutes a cluster boundary.
```

Now let's try it out and see if it works any better on our smiley face.

In [ ]:
from sklearn.cluster import OPTICS

In [ ]:
# Initialise and fit the OPTICS clustering model with steepness xi=0.05 and min_cluster_size=0.05
op = OPTICS(xi=0.05, min_cluster_size=.05).fit(X)

# Extract the cluster labels assigned by OPTICS.
labels = op.labels_

# Calculate the number of unique clusters, excluding noise (-1 label).
n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)

# Calculate the number of noise points (-1 labels).
n_noise_ = list(labels).count(-1)

# Print the estimated number of clusters and noise points.
print('Estimated number of clusters: %d' % n_clusters_)
print('Estimated number of noise points: %d' % n_noise_)

# Get the unique labels from the clustered data.
unique_labels = np.unique(labels)

# Generate a list of colours for each unique label.
colors = [plt.cm.Spectral(each)
          for each in np.linspace(0, 1, len(unique_labels))]


plt.figure(figsize=(6,6))

# Iterate through each unique cluster label (excluding noise which is -1), and its corresponding colour.
for klass, color in zip(unique_labels[1:], colors[1:]):

    # Select the data points belonging to the current cluster.
    Xk = X[op.labels_ == klass]
    # Create a scatter plot of the data points in the current cluster.
    plt.scatter(Xk[:, 0], Xk[:, 1], 60, np.array([color,]),
                'o', edgecolors='k',linewidths=1)

# Plot the noise points (labeled as -1) as black crosses.
plt.plot(X[op.labels_ == -1, 0], X[op.labels_ == -1, 1], 'k+', ls = 'None')
plt.title('Estimated number of clusters: %d' % n_clusters_)
plt.show()

<div style="background-color:#C2F5DD">

## Exercise 3
1. Use `DBSCAN` and vary the distance paramenter `eps` (0.2, 0.25, 0.3, 0.35) and check how the results of DBSCAN change.
2. Now keep `eps` fixed to what you think is an optimal value and vary the minimum sample size from 2 to 8. 
3. Comment on the results.
4. Now try out `Optics` xi=0.2.
5. Comment on the results.


My final note here is that building and evaluating clustering schemes is tricky. For our clustering to make sense, we need to have a good knowledge of the structure of our data, but unfortunately, this is often what we'd like to learn by building the clustering in first place. I don't have a lot of wisdom to share; just this word of caution.

## Dimensionality Reduction

Principal Component Analysis (PCA) and similar algorithms are used for dimensionality reduction in data-intensive sciences. 

The main goal of linear PCA is to find the most representative linear combinations of features, so that each element of a data set can be expressed as the superposition (sum) of some salient vectors in feature space (they don't need to be elements of a data set. In the simplest linear PCA, the principal components are the eigenvectors of the covariance matrix of the data set.

If the number of components is very low (e.g. 2 or 3, PCA or other Dimensionality Reduction methods allow one to visualise a high-dimensional data set as a 2D or 3D plot. Scikit-learn has methods to compute PCA and several variants. Classic PCA has tough complexity: $\mathcal{O}[N^3].$ 

Dataset taken from: 
https://ogrisel.github.io/scikit-learn.org/sklearn-tutorial/tutorial/astronomy/dimensionality_reduction.html#sdss-spectral-data

In [ ]:
data = np.load('spec4000_corrected.npz')

In [ ]:
wavelengths = data['wavelengths']
X = data['X']
y = data['y']
labels = data['labels'].astype('str')

In [ ]:
X.shape

In [ ]:
y

In [ ]:
labels #we don't really care about these; they are only used in the next cell to show some example spectra.

We can plot some representative examples from each class, just to have a sense of what kind of spectra are in the data set.


In [ ]:
plt.figure(figsize=(10,5))

for i_class in (2, 3, 4, 5, 6):
    i = np.where(y == i_class)[0][0]
    l = plt.plot(wavelengths, X[i] + 20 * i_class)
    c = l[0].get_color()
    plt.text(6800, 2 + 20 * i_class, labels[i_class], color=c)

plt.subplots_adjust(hspace=0)
plt.xlabel('wavelength (Angstroms)')
plt.ylabel('flux + offset')
plt.title('Sample of Spectra');

Our original data set has 4,000 objects and 1,000 features. 

We will try to represent it with a variable amount of components.

In [ ]:
# Perform Principal Component Analysis (PCA)

# Use a StandardScaler to standardise the data.
# It's crucial to centre the data before PCA for optimal results.
scaler = preprocessing.StandardScaler()
Xn = scaler.fit_transform(X)

# Initialise PCA with 10 principal components.
# random_state=0 ensures the reproducibility of the results.
pca_10 = decomposition.PCA(n_components=10, random_state=0)

# Apply PCA with 10 components to the standardised data Xn.
# fit_transform calculates the principal components and transforms the data into the new feature space.
# X_proj_10 is the projected dataset, now with 10 features.
X_proj_10 = pca_10.fit_transform(Xn)


In [ ]:
#----------------------------------------------------------------------
#
#  plot PCA eigenspectra
#

plt.figure()

l = plt.plot(wavelengths, pca_10.mean_ - 0.15)
c = l[0].get_color()
plt.text(7000, -0.1, "mean", color=c) 

# In linear PCA, the first eigenvector is always the mean, 
# and the first n components are always the same

for i in range(4):
    
    l = plt.plot(wavelengths, pca_10.components_[i] + 0.15 * i)
        
    c = l[0].get_color()
    
    plt.text(7000, 0.05  + 0.15 * i, "component %i" % (i + 1), color=c)

    plt.ylim(-0.2, 0.6)
    
plt.xlabel('wavelength (Angstroms)')
plt.ylabel('scaled flux + offset')
plt.title('Mean Spectrum and Eigen-spectra')

plt.show()


<div style="background-color:#C2F5DD">

## Exercise 4
1. Now repeat the same procedure as above but for 50, 100 and 1000 components.
2. We can estimate the contribution of each component by using the property "explained variance ratio" (accessed as `pca_10.explained_variance_ratio`), these are simply the eigenvalues of the covariance matrix, and give an idea of how much each component contributes to the total variance in the data. Plot the cumulative sum for all four cases and check how many components do you think would be enough to explain the variance?
3. The `inverse_transform()` method of the pca object is used to reconstruct the data from its projected representation (eg `pca_10.inverse_transform(X_proj_10)`) back into the original feature space. Plot the relative difference between the reconstructed representations and the original ones for all 4 PCAs. Do you still agree with your answer to question 2?

### Let's look at a dataset of images

In [ ]:
#Takes < 1 minute, let's use non-bubbled images

images = []
for i in range(200):
    img =skimage.io.imread('Images/Image_'+str(i)+'.png')
    img_resized = resize(img,(100,100))
    length = np.prod(img_resized.shape)
    img_resized = np.reshape(img_resized,length)
    images.append(img_resized)
    
images = np.vstack(images)

In [ ]:
fig, axes = plt.subplots(ncols= 5, nrows = 5,figsize=(50,50))

ax = axes.ravel()

for i in range(ax.shape[0]):

    img = skimage.io.imread('Images/Image_'+str(i)+'.png')
    ax[i].imshow(img, cmap='gray')
    ax[i].set_xticks([])
    ax[i].set_yticks([])


Here, we do the PCA decomposition on each of the RGB channels separately (even if it's not optimal). 
The line below reshapes the images and selects only the first colour (red).

In [ ]:
r_images = images.reshape(200, -1,  3)[:,:,0]

In [ ]:
r_images.shape

In [ ]:
#Calling PCA on images

estimator = decomposition.PCA(n_components=100)

r_images_PCA = estimator.fit_transform(r_images)

In [ ]:
#This tell us about the dimensionality reduction we have achieved.

r_images_PCA.shape

In [ ]:
components = estimator.components_

Plot the first 50 components. From this plot, would you guess an optimal # of components?

In [ ]:
fig, axes = plt.subplots(5, 10, figsize=(12, 6),
                         subplot_kw={'xticks':[], 'yticks':[]},
                         gridspec_kw=dict(hspace=0.1, wspace=0.1))
for i, ax in enumerate(axes.flat):
    ax.imshow((estimator.components_[i].reshape(100, 100)), cmap='gray')

We can use the explained variance ratio to see if there is an obvious optimal number of components.

In [ ]:
plt.plot(np.cumsum(estimator.explained_variance_ratio_))
plt.xlabel('number of components')
plt.ylabel('cumulative explained variance');

**Note how different this plot is, compared to the case above, with the variance more equally distributed among many components. We need 60-70 components just to retain ~ 90% of the variance.**

<div style="background-color:#C2F5DD">

## Exercise 5 - OPTIONAL
1. Now reconstruct the images using the inverse transform.
2. Plot the first 10 reconstructed images alongside the original (use a gray scale) and compare them. Visual comparisons could be done with image differences. Quantitative comparisons will be useful too.
3. Repeat the above steps with 200 components. Are the results any better?